In [0]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import yaml
from pyspark.sql import functions as F
from nyc311.schema import bronze_schema

with open("../conf/config.yaml") as f:
    cfg = yaml.safe_load(f)

catalog, schema = cfg["catalog"], cfg["schema"]
landing = f"/Volumes/{catalog}/{schema}/{cfg['landing_volume']}"

df = (spark.read
        .schema(bronze_schema())
        .json(f"{landing}/ingest_date=*/")
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path")))

(df.write
   .format("delta")
   .mode("append")
   .saveAsTable(f"{catalog}.{schema}.bronze_complaints"))